## 工具的应用案例

### 1. 使用 args_schema

In [ ]:
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from rich import print as rprint
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool

load_dotenv(override=True)
ZHIPUAI_API_KEY = os.getenv("ZHIPUAI_API_KEY")
ZHIPUAI_API_BASE_URL = os.getenv("ZHIPUAI_API_BASE_URL")

# 获取大模型
model = init_chat_model(
    model='glm-5.3',
    model_provider="openai",
    api_key=ZHIPUAI_API_KEY,
    base_url=ZHIPUAI_API_BASE_URL
)

In [ ]:
from pydantic import BaseModel, Field

class WeacherSchema(BaseModel):
    city: str = Field(default="北京", description="具体的城市名称")
    if_forecast: bool = Field(description="是否包含明天天气")

# 定义工具
@tool(
        "get_weacher_and_forecast", 
        description="查询当日的天气, 可以包含明天的天气预报",
        args_schema=WeacherSchema
      )
def get_weather(city:str, if_forecast:bool):
    res = f"{city}今天天气不错"
    if if_forecast:
        res += "\n明天下雨"
    return res

rprint(convert_to_openai_tool(get_weather))
get_weather_info = convert_to_openai_tool(get_weather)
rprint(type(get_weather_info))

In [ ]:
# 1. 将工具绑定到模型上
model_with_tools = model.bind_tools([get_weather])

# 2. 维护一个消息列表
messages = [
    HumanMessage("今天杭州的天气怎么样? 明天呢?")
]

# 3. 调用模型, 得到相应: AIMessage
response = model_with_tools.invoke(messages)
# rprint(response)

messages.append(response)

# 4. 获取响应中的 tool_calls 字段信息
tool_calls = response.tool_calls
rprint(tool_calls)

for tool_call in tool_calls:
    if tool_call["name"] == get_weather_info["function"]["name"]:
        # 5. 调用工具 (因为大模型不能直接调用工具, 所以此时我们主动让工具调用执行)
        tool_message = get_weather.invoke(tool_call)
        messages.append(tool_message)

# 6. 调用模型, 得到 AIMessage
final_response = model.invoke(messages)
rprint(final_response)

# 7. 添加到消息列表中
messages.append(final_response)

# 8. 遍历消息列表
for msg in messages:
    msg.pretty_print()

### 2. 撰写 docstring

In [ ]:
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from rich import print as rprint
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool

load_dotenv(override=True)
ZHIPUAI_API_KEY = os.getenv("ZHIPUAI_API_KEY")
ZHIPUAI_API_BASE_URL = os.getenv("ZHIPUAI_API_BASE_URL")

# 获取大模型
model = init_chat_model(
    model='glm-5.3',
    model_provider="openai",
    api_key=ZHIPUAI_API_KEY,
    base_url=ZHIPUAI_API_BASE_URL
)

In [ ]:
# 定义工具
@tool(
        "get_weacher_and_forecast", 
        parse_docstring=True
      )
def get_weather(city:str, if_forecast:bool):
    """
        查询当日的天气, 可以包含明天的天气预报

        Args:
            city: 城市名称
            if_forecast: 是否包含明天天气
    """
    res = f"{city}今天天气不错"
    if if_forecast:
        res += "\n明天下雨"
    return res

rprint(convert_to_openai_tool(get_weather))
get_weather_info = convert_to_openai_tool(get_weather)
rprint(type(get_weather_info))

In [ ]:
# 1. 将工具绑定到模型上
model_with_tools = model.bind_tools([get_weather])

# 2. 维护一个消息列表
messages = [
    HumanMessage("今天杭州的天气怎么样? 明天呢?")
]

# 3. 调用模型, 得到相应: AIMessage
response = model_with_tools.invoke(messages)
# rprint(response)

messages.append(response)

# 4. 获取响应中的 tool_calls 字段信息
tool_calls = response.tool_calls
rprint(tool_calls)

for tool_call in tool_calls:
    if tool_call["name"] == get_weather_info["function"]["name"]:
        # 5. 调用工具 (因为大模型不能直接调用工具, 所以此时我们主动让工具调用执行)
        tool_message = get_weather.invoke(tool_call)
        messages.append(tool_message)

# 6. 调用模型, 得到 AIMessage
final_response = model.invoke(messages)
rprint(final_response)

# 7. 添加到消息列表中
messages.append(final_response)

# 8. 遍历消息列表
for msg in messages:
    msg.pretty_print()

### 3. 多工具调用

In [30]:
# 大模型调用工具是单次推理, 即每次运行调用一个工具, 当调用多个工具时, 需要用户自己管理多个调用循环

# 1. 定义工具
# 定义股票查询工具
@tool('aaa', parse_docstring=True)
def get_stock_price(company:str, timeframe:str = "today") -> str:
    """
    获取指定公司的股票价格信息

    Args:
        company: 公司名称(如: 苹果, 微软, 谷歌)
        timeframe: 时间范围(today-今天, week-本周, month-本月)
    """
    # 模拟股票数据
    mock_data = {
        "苹果": {"today": 185.20, "week": 183.40, "month": 180.75},
        "微软": {"today": 7666.20, "week": 666.40, "month": 888.75},
        "谷歌": {"today": 23422.20, "week": 888.40, "month": 777.75}
    }
    if company in mock_data:
        price = mock_data[company].get(timeframe, "未知时间范围")
        return f"{company} {timeframe} 价格: {price}美元"
    else:
        return f"未找到股票代码{company}的数据"

# 定义新闻搜索工具
@tool(parse_docstring=True)
def search_news(company: str) -> str:
    """
    搜索指定公司的财经新闻

    Args:
        company: 公司名称
    """
    # 模拟新闻数据
    mock_news = {
        "苹果": [
            "苹果哈哈",
            "苹果"
        ],
        "微软": [
            "微软"
        ],
        "谷歌": [
            "google"
        ]
    }
    news_list = mock_news.get(company, [f"未找到{company}的相关新闻"])
    return "\n".join(news_list)

# 2. 初始化模型并绑定工具
tools = [get_stock_price, search_news]
model_with_tools = model.bind_tools(tools)

messages_list = []
human_message = HumanMessage(content="苹果公司今天的股价是多少?最近有什么新闻?")
# human_message = HumanMessage(content="比较一下微软和苹果股价") 
# human_message = HumanMessage(content="腾讯最近有什么新闻") 
# human_message = HumanMessage(content="海水为什么是咸的") 
messages_list.append(human_message)

# 3. 调用工具
# 方式一
# while True:
#     response = model_with_tools.invoke(messages_list)
#     rprint(response)
#     messages_list.append(response)

#     # 如果模型不需要调用工具, 直接退出循环
#     if not response.tool_calls:
#         print("没有工具调用, 直接返回答案")
#         break

#     # 如果有调用工具, 处理工具响应
#     # 4. 开发者根据模型的相应, 调用工具并获取结果
#     for tool_call in response.tool_calls:
#         if tool_call["name"] == get_stock_price.name:
#             # 5. 调用工具 (因为大模型不能直接调用工具, 所以此时我们主动让工具调用执行)
#             stock_result = get_stock_price.invoke(tool_call)
#             messages_list.append(tool_message)
#         if tool_call["name"] == search_news.name:
#             # 5. 调用工具 (因为大模型不能直接调用工具, 所以此时我们主动让工具调用执行)
#             stock_result = get_stock_price.invoke(tool_call)
#             messages_list.append(tool_message)

# rprint(response.content)

# 方式二
response = model_with_tools.invoke(messages_list)
rprint(response)
messages_list.append(response)

for tool_call in response.tool_calls:
    if tool_call["name"] == get_stock_price.name:
        # 5. 调用工具 (因为大模型不能直接调用工具, 所以此时我们主动让工具调用执行)
        stock_result = get_stock_price.invoke(tool_call)
        messages_list.append(tool_message)
    elif tool_call["name"] == search_news.name:
        # 5. 调用工具 (因为大模型不能直接调用工具, 所以此时我们主动让工具调用执行)
        stock_result = get_stock_price.invoke(tool_call)
        messages_list.append(tool_message)
    else:
        raise Exception("不存在的工具")

final_response = model.invoke(messages_list)
messages_list.append(final_response)

for msg in messages_list:
    msg.pretty_print()


AIMessage(
    content='',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 64,
            'prompt_tokens': 270,
            'total_tokens': 334,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 36,
                'rejected_prediction_tokens': None
            },
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
        },
        'model_provider': 'openai',
        'model_name': 'glm-5.3',
        'system_fingerprint': None,
        'id': '2026111111354829aef275f85f45a6',
        'finish_reason': 'tool_calls',
        'logprobs': None
    },
    id='lc_run--01a1c8ad-142c-7b20-9871-fef8d313d75f-0',
    tool_calls=[
        {
            'name': 'aaa',
            'args': {'company': '苹果', 'timeframe': 'today'},
            'id': 'call_-7086080311888441489',
            'type': 'tool_call'
        },
        {
            'name': 'search_news',
            'args': {'company': '苹果'},
            'id': 'call_-7086080311888441488',
            'type': 'tool_call'
        }
    ],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 270,
        'output_tokens': 64,
        'total_tokens': 334,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {'reasoning': 36}
    }
)

================================ Human Message =================================

苹果公司今天的股价是多少?最近有什么新闻?
================================== Ai Message ==================================
Tool Calls:
  aaa (call_-7086080311888441489)
 Call ID: call_-7086080311888441489
  Args:
    company: 苹果
    timeframe: today
  search_news (call_-7086080311888441488)
 Call ID: call_-7086080311888441488
  Args:
    company: 苹果
================================= Tool Message =================================
Name: get_weacher_and_forecast

杭州今天天气不错
明天下雨
================================= Tool Message =================================
Name: get_weacher_and_forecast

杭州今天天气不错
明天下雨
================================== Ai Message ==================================

抱歉，我目前无法获取苹果公司的实时股价和最新新闻。作为AI助手，我没有实时数据访问权限，无法查询当前的市场信息。

**建议您通过以下途径获取准确信息：**

📈 **查询股价：**
- 财经网站：雅虎财经、谷歌财经、东方财富
- 证券交易App：同花顺、雪球、富途牛牛
- 苹果股票代码：**AAPL**（纳斯达克上市）

📰 **查看新闻：**
- 苹果官网新闻room
- 彭博社、路透社等财经媒体
- 各大科技媒体（如36氪、爱范儿）

**我可以帮助您的方面：**
- 分析苹果公司的商业模